In [ ]:
"""
==========================================
Image Captioning with GIT (CS6350 - TPA19) 
==========================================
Self-contained counterpart to BLIP.ipynb. Uses the SAME COCO data, the
SAME train/val split files (created by BLIP's prepare_data), and the
SAME held-out test set (val2017), so results are directly comparable.
 
Stages:
    python GIT.py --stage prepare_data
    python GIT.py --stage evaluate --checkpoint microsoft/git-base-coco     # pretrained baseline
    python GIT.py --stage train --run_name git_1epoch --epochs 1 --max_train_samples 40000
    python GIT.py --stage evaluate --checkpoint ./checkpoints/git_1epoch/epoch0
    python GIT.py --stage qualitative --checkpoint ./checkpoints/git_1epoch/epoch0
    python GIT.py --stage zero_shot --checkpoint ./checkpoints/git_1epoch/epoch0 --image_dir ./image_dir
 
Or from a notebook:
    import GIT
    GIT.train(run_name="git_1epoch", num_epochs=1)
    GIT.evaluate("./checkpoints/git_1epoch/epoch0")
 
Every run writes to its own folder:
    checkpoints/<run_name>/epoch<N>/
    outputs/<run_name>_epoch<N>/   (predictions.json, metrics.json, grids)
======================================================================
"""
 
import os
import json
import time
import random
import argparse
import textwrap
from PIL import Image, ImageOps
from pathlib import Path
 
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
 
# ----------------------------------------------------------------------
# 0. CONFIG  (paths/splits match BLIP.ipynb so both models share data)
# ----------------------------------------------------------------------
CONFIG = {
    "model_name": "microsoft/git-base",
    "data_root": "./coco_data",
    "checkpoint_dir": "./checkpoints",
    "output_dir": "./outputs",
 
    "train_ann": "annotations/captions_train2017.json",
    "train_img_dir": "train2017",
    "train_split_ann": "annotations/captions_train_split.json",
    "val_split_ann": "annotations/captions_val_split.json",
    "val_split_ratio": 0.1,
    "split_seed": 42,
 
    # val2017 = held-out TEST set. Never used for training or tuning.
    "test_ann": "annotations/captions_val2017.json",
    "test_img_dir": "val2017",
    "max_test_samples": 5000,
 
    "max_train_samples": 100000,   # GIT steps are faster than BLIP's; adjust to budget
    "max_val_samples": 500,       # val-split subsample for the post-training sanity loss
 
    "batch_size": 16,
    "num_epochs": 1,
    "lr": 5e-5,
    "warmup_ratio": 0.05,
    "max_caption_len": 40,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "n_trials": 16,
    "tune_max_steps": 200,        # cap steps per trial
    "max_tune_val_samples": 500,  # val-split subsample scored per trial
}
 
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

In [ ]:
def load_model_and_processor(path):
    processor = AutoProcessor.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path).to(CONFIG["device"])
    return model, processor
 
 
def get_output_dir(checkpoint_path):
    """
    Maps a checkpoint path to its own output folder, e.g.
      ./checkpoints/git_1epoch/epoch0 -> outputs/git_1epoch_epoch0/
      microsoft/git-base-coco         -> outputs/git-base-coco/
    """
    if checkpoint_path is None:
        tag = "git-base"
    else:
        p = Path(checkpoint_path)
        try:
            tag = "_".join(p.relative_to(CONFIG["checkpoint_dir"]).parts)
        except ValueError:
            tag = p.name
    out_dir = Path(CONFIG["output_dir"]) / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

In [ ]:
# ----------------------------------------------------------------------
# 1. DATASET
# ----------------------------------------------------------------------
class GitCaptionDataset(Dataset):
    """
    COCO-format captions flattened to (image, one caption) pairs.
    Padding positions are set to -100 in the labels so they are ignored
    by the loss.
    """
 
    def __init__(self, ann_path, img_dir, processor, max_samples=None):
        with open(ann_path, "r") as f:
            coco = json.load(f)
 
        id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        self.samples = [
            {
                "image_path": os.path.join(img_dir, id_to_file[a["image_id"]]),
                "caption": a["caption"].strip(),
            }
            for a in coco["annotations"]
            if a["image_id"] in id_to_file
        ]
        random.shuffle(self.samples)
        if max_samples:
            self.samples = self.samples[:max_samples]
        self.processor = processor
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        enc = self.processor(
            images=image,
            text=item["caption"],
            padding="max_length",
            truncation=True,
            max_length=CONFIG["max_caption_len"],
            return_tensors="pt",
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        enc["labels"] = labels
        return enc
 
 
def split_train_val():
    """Same split logic as BLIP.ipynb (same seed -> identical split)."""
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["train_ann"], "r") as f:
        coco = json.load(f)
 
    image_ids = [img["id"] for img in coco["images"]]
    rng = random.Random(CONFIG["split_seed"])
    rng.shuffle(image_ids)
 
    n_val = int(len(image_ids) * CONFIG["val_split_ratio"])
    val_ids = set(image_ids[:n_val])
    train_ids = set(image_ids[n_val:])
 
    train_split = {
        "images": [i for i in coco["images"] if i["id"] in train_ids],
        "annotations": [a for a in coco["annotations"] if a["image_id"] in train_ids],
    }
    val_split = {
        "images": [i for i in coco["images"] if i["id"] in val_ids],
        "annotations": [a for a in coco["annotations"] if a["image_id"] in val_ids],
    }
 
    out_dir = root / "annotations"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(root / CONFIG["train_split_ann"], "w") as f:
        json.dump(train_split, f)
    with open(root / CONFIG["val_split_ann"], "w") as f:
        json.dump(val_split, f)
    print(f"Train split: {len(train_split['images'])} images; "
          f"val split: {len(val_split['images'])} images")

In [ ]:
def prepare_data():
    root = Path(CONFIG["data_root"])
    assert (root / CONFIG["train_ann"]).exists() and (root / CONFIG["test_ann"]).exists(), (
        "COCO annotations not found under ./coco_data. Download COCO 2017 first."
    )
    print("Data check passed.")
    if (root / CONFIG["train_split_ann"]).exists() and (root / CONFIG["val_split_ann"]).exists():
        print("Train/val split already exists (shared with BLIP) -- skipping.")
    else:
        print("Creating train/val split...")
        split_train_val()
 
 
# ----------------------------------------------------------------------
# 2. TRAINING
# ----------------------------------------------------------------------
def compute_loss(model, batch):
    """
    Next-token cross-entropy with the label shift done exactly ONCE, here.
    We deliberately do not pass labels= to the model: some transformers
    versions shift GIT's labels twice inside forward(), which trains the
    model to predict the token two positions ahead and wrecks it.
    GIT's logits cover [image tokens + text tokens]; keep the text part.
    """
    out = model(
        pixel_values=batch["pixel_values"],
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )
    n_img = out.logits.shape[1] - batch["input_ids"].shape[1]
    text_logits = out.logits[:, n_img:, :]
    return F.cross_entropy(
        text_logits[:, :-1].reshape(-1, text_logits.shape[-1]).float(),
        batch["labels"][:, 1:].reshape(-1),
        ignore_index=-100,
    )
 
 
def check_loss(model_name=None, n_batches=4, batch_size=16):
    """
    One-minute sanity test (no training). On the pretrained COCO model,
    compare the loss HuggingFace computes with ours. A healthy model gives
    roughly 2-3 for compute_loss. If HF's number is far higher (~7+), the
    built-in loss is mis-shifting the labels on your transformers version.
    """
    device = CONFIG["device"]
    model, processor = load_model_and_processor(model_name or CONFIG["model_name"])
    root = Path(CONFIG["data_root"])
    ds = GitCaptionDataset(root / CONFIG["val_split_ann"], root / CONFIG["train_img_dir"],
                           processor, max_samples=n_batches * batch_size)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    hf, ours = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            hf.append(model(pixel_values=batch["pixel_values"], input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"],
                            labels=batch["labels"]).loss.item())
            ours.append(compute_loss(model, batch).item())
    print(f"HF built-in loss : {sum(hf) / len(hf):.3f}")
    print(f"compute_loss     : {sum(ours) / len(ours):.3f}")

In [ ]:
def run_validation_loss(model, val_loader, device):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            total += compute_loss(model, batch).item()
            n += 1
    return total / max(n, 1)
 
 
def tune_hyperparameters(n_trials=None):
    """
    Optuna search over lr / batch_size / warmup_ratio for GIT. Each trial
    trains for at most tune_max_steps steps on the TRAIN split, then is
    scored by loss on a subsample of the VAL split (never val2017).
    Best params are saved to outputs/git_best_hparams.json (separate from
    BLIP's best_hparams.json), which train() loads automatically.
    """
    import optuna
 
    n_trials = n_trials or CONFIG["n_trials"]
    device = CONFIG["device"]
    use_amp = device == "cuda"
    root = Path(CONFIG["data_root"])
 
    def objective(trial):
        lr = trial.suggest_float("lr", 1e-6, 1e-4, log=True)
        batch_size = trial.suggest_categorical("batch_size", [8, 16])
        warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
 
        model = optimizer = scheduler = train_loader = val_loader = None
        t_trial = time.time()
        try:
            model, processor = load_model_and_processor(CONFIG["model_name"])
            # pool just big enough for tune_max_steps batches
            pool = CONFIG["tune_max_steps"] * batch_size
            train_ds = GitCaptionDataset(
                root / CONFIG["train_split_ann"], root / CONFIG["train_img_dir"],
                processor, max_samples=pool)
            val_ds = GitCaptionDataset(
                root / CONFIG["val_split_ann"], root / CONFIG["train_img_dir"],
                processor, max_samples=CONFIG["max_tune_val_samples"])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
 
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
            total_steps = min(len(train_loader), CONFIG["tune_max_steps"])
            scheduler = get_linear_schedule_with_warmup(
                optimizer, int(warmup_ratio * total_steps), max(total_steps, 1))
            scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
 
            print(f"[trial {trial.number}] starting: lr={lr:.2e} batch={batch_size} "
                  f"warmup={warmup_ratio:.3f} (setup took {time.time() - t_trial:.0f}s)")
            t_train = time.time()
            model.train()
            for step, batch in enumerate(train_loader):
                if step >= CONFIG["tune_max_steps"]:
                    break
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                with torch.amp.autocast("cuda", enabled=use_amp):
                    loss = compute_loss(model, batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                if step % 50 == 0:
                    print(f"[trial {trial.number}] step {step}/{CONFIG['tune_max_steps']} "
                          f"loss {loss.item():.3f} | {time.time() - t_train:.0f}s")
 
            return run_validation_loss(model, val_loader, device)
 
        except torch.cuda.OutOfMemoryError:
            print(f"Trial {trial.number} OOM (batch={batch_size}) -- pruning.")
            raise optuna.exceptions.TrialPruned()
        finally:
            del model, optimizer, scheduler, train_loader, val_loader
            if use_amp:
                torch.cuda.empty_cache()
 
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    print("Best hyperparameters:", study.best_params)
    print("Best val loss:", study.best_value)
 
    out_dir = Path(CONFIG["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "git_best_hparams.json", "w") as f:
        json.dump(study.best_params, f, indent=2)
    print(f"Saved to {out_dir / 'git_best_hparams.json'}")
    return study.best_params
    

In [ ]:
def train(run_name=None, num_epochs=None, max_train_samples=None,
          lr=None, batch_size=None, warmup_ratio=None, model_name=None):
    """
    Fine-tunes GIT on the TRAIN split. Each call saves to its own folder:
        checkpoints/<run_name>/epoch<N>/
    so nothing is ever overwritten. run_name defaults to a timestamp.
    Any of lr/batch_size/warmup_ratio not passed explicitly are taken from
    outputs/git_best_hparams.json (if tune_hyperparameters() was run),
    otherwise from CONFIG.
    """
    tuned = {}
    tuned_path = Path(CONFIG["output_dir"]) / "git_best_hparams.json"
    if tuned_path.exists():
        with open(tuned_path) as f:
            tuned = json.load(f)
        print(f"Loaded tuned hyperparameters from {tuned_path}: {tuned}")
 
    run_name = run_name or time.strftime("git_%Y%m%d_%H%M%S")
    num_epochs = num_epochs or CONFIG["num_epochs"]
    max_train_samples = max_train_samples or CONFIG["max_train_samples"]
    lr = lr or tuned.get("lr", CONFIG["lr"])
    batch_size = batch_size or tuned.get("batch_size", CONFIG["batch_size"])
    if warmup_ratio is None:
        warmup_ratio = tuned.get("warmup_ratio", CONFIG["warmup_ratio"])
    model_name = model_name or CONFIG["model_name"]
 
    device = CONFIG["device"]
    use_amp = device == "cuda"
    print(f"Run '{run_name}' | model={model_name} | epochs={num_epochs} | "
          f"lr={lr} | batch={batch_size} | samples={max_train_samples}")
 
    model, processor = load_model_and_processor(model_name)
 
    root = Path(CONFIG["data_root"])
    train_ds = GitCaptionDataset(
        root / CONFIG["train_split_ann"], root / CONFIG["train_img_dir"],
        processor, max_samples=max_train_samples,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    print(f"Dataset: {len(train_ds)} samples, {len(train_loader)} batches/epoch")
 
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(warmup_ratio * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
 
    run_dir = Path(CONFIG["checkpoint_dir"]) / run_name
    model.train()
    for epoch in range(num_epochs):
        print(f"--- Epoch {epoch} ---")
        running = 0.0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = compute_loss(model, batch)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
 
            running += loss.item()
            if step % 50 == 0:
                print(f"epoch {epoch} step {step}/{len(train_loader)} "
                      f"loss {running / (step + 1):.4f}")
 
        ckpt = run_dir / f"epoch{epoch}"
        ckpt.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt)
        processor.save_pretrained(ckpt)
        print(f"Saved checkpoint: {ckpt}")
        model.train()
 
    # Sanity check only -- the reported metric comes from evaluate() on val2017.
    val_ds = GitCaptionDataset(
        root / CONFIG["val_split_ann"], root / CONFIG["train_img_dir"],
        processor, max_samples=CONFIG["max_val_samples"],
    )
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    print(f"Post-training val-split loss (sanity check): "
          f"{run_validation_loss(model, val_loader, device):.4f}")
    return run_dir
 
 
# ----------------------------------------------------------------------
# 3. INFERENCE
# ----------------------------------------------------------------------
def generate_captions(model, processor, image_paths, device, batch_size=16, num_beams=4):
    model.eval()
    results = {}
    n_batches = (len(image_paths) + batch_size - 1) // batch_size
    t0 = time.time()
    with torch.no_grad():
        for b, i in enumerate(range(0, len(image_paths), batch_size)):
            batch_paths = image_paths[i:i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)
            out_ids = model.generate(
                pixel_values=pixel_values,
                max_length=CONFIG["max_caption_len"],
                num_beams=num_beams,
                use_cache=True,
            )
            if b % 10 == 0:
                el = time.time() - t0
                eta = el / (b + 1) * (n_batches - b - 1)
                print(f"  generation batch {b + 1}/{n_batches} | {el:.0f}s elapsed | ~{eta / 60:.1f} min left")
            captions = processor.batch_decode(out_ids, skip_special_tokens=True)
            for p, c in zip(batch_paths, captions):
                results[p] = c.strip()
    return results

In [ ]:
# ----------------------------------------------------------------------
# 4. EVALUATION on the held-out TEST set (val2017)
# ----------------------------------------------------------------------
def evaluate(checkpoint_path=None, num_beams=4):
    """
    BLEU-4 (smoothed), METEOR, ROUGE-L via nltk/rouge-score, CIDEr via
    pycocoevalcap. Same metric code as BLIP.ipynb's evaluate().
    checkpoint_path=None or a hub name evaluates the pretrained model.
    """
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
 
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    print(f"Device: {device} | loading: {model_path}")
    model, processor = load_model_and_processor(model_path)
 
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
 
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs_by_image = {}
    for a in coco["annotations"]:
        refs_by_image.setdefault(a["image_id"], []).append(a["caption"].strip())
 
    image_ids = list(refs_by_image.keys())[: CONFIG["max_test_samples"]]
    image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in image_ids]
    print(f"Evaluating on {len(image_paths)} held-out TEST images (val2017)...")
 
    preds = generate_captions(model, processor, image_paths, device, num_beams=num_beams)
    print("Caption generation done. Computing metrics...")
 
    smoothie = SmoothingFunction().method4
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
 
    bleu_scores, meteor_scores, rouge_scores = [], [], []
    coco_gts, coco_res = {}, {}
    for img_id, path in zip(image_ids, image_paths):
        hyp = preds[path]
        refs = refs_by_image[img_id]
        ref_tokens = [r.split() for r in refs]
 
        bleu_scores.append(sentence_bleu(ref_tokens, hyp.split(), smoothing_function=smoothie))
        meteor_scores.append(meteor_score(ref_tokens, hyp.split()))
        rouge_scores.append(max(scorer.score(r, hyp)["rougeL"].fmeasure for r in refs))
        coco_gts[img_id] = refs
        coco_res[img_id] = [hyp]
 
    metrics = {
        "BLEU-4": sum(bleu_scores) / len(bleu_scores),
        "METEOR": sum(meteor_scores) / len(meteor_scores),
        "ROUGE-L": sum(rouge_scores) / len(rouge_scores),
    }
    print(f"BLEU-4 (avg, smoothed): {metrics['BLEU-4']:.4f}")
    print(f"METEOR (avg):           {metrics['METEOR']:.4f}")
    print(f"ROUGE-L (avg F1):       {metrics['ROUGE-L']:.4f}")
 
    try:
        from pycocoevalcap.cider.cider import Cider
        cider_score, _ = Cider().compute_score(coco_gts, coco_res)
        metrics["CIDEr"] = cider_score
        print(f"CIDEr:                  {cider_score:.4f}")
    except ImportError:
        print("pycocoevalcap not installed -- skipping CIDEr.")
 
    out_dir = get_output_dir(checkpoint_path)
    with open(out_dir / "predictions.json", "w") as f:
        json.dump(preds, f, indent=2)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Predictions + metrics saved to {out_dir}")
    return metrics

In [ ]:
# ----------------------------------------------------------------------
# 5. QUALITATIVE ANALYSIS (fixed sample -> same images as BLIP's grid)
# ----------------------------------------------------------------------
def qualitative_analysis(checkpoint_path=None, n_samples=8):
    import matplotlib.pyplot as plt
 
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    model, processor = load_model_and_processor(model_path)
 
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
 
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs_by_image = {}
    for a in coco["annotations"]:
        refs_by_image.setdefault(a["image_id"], []).append(a["caption"].strip())
 
    # Same seed + same key list as BLIP.ipynb's qualitative_analysis, so
    # both models are shown on identical images.
    sample_ids = random.Random(CONFIG["seed"]).sample(list(refs_by_image.keys()), n_samples)
    image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in sample_ids]
    preds = generate_captions(model, processor, image_paths, device)
 
    fig, axes = plt.subplots(2, n_samples // 2, figsize=(4 * n_samples // 2, 8))
    for ax, img_id, path in zip(axes.flat, sample_ids, image_paths):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
        ax.set_title(f"Pred: {preds[path]}\nGT: {refs_by_image[img_id][0]}",
                     fontsize=8, wrap=True)
 
    out_dir = get_output_dir(checkpoint_path)
    fig.tight_layout()
    fig.savefig(out_dir / "qualitative_grid.png", dpi=150)
    print(f"Saved qualitative grid to {out_dir / 'qualitative_grid.png'}")
 
 
# ----------------------------------------------------------------------
# 6. ZERO-SHOT / OUT-OF-DOMAIN
# ----------------------------------------------------------------------
def zero_shot(image_dir, checkpoint_path=None, ncols=3, max_images=6):
    """
    Zero-shot captioning on our own images (out-of-domain). Shows each image with its
    caption in a uniform grid, drawn at printed size (6.5 in wide) for the report.
    """
    import matplotlib.pyplot as plt

    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    model, processor = load_model_and_processor(model_path)

    image_paths = [
        str(p) for p in sorted(Path(image_dir).glob("*"))
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ][:max_images]
    preds = generate_captions(model, processor, image_paths, device)
    for p, c in preds.items():
        print(f"{Path(p).name}: {c}")

    out_dir = get_output_dir(checkpoint_path)
    with open(out_dir / "zero_shot_predictions.json", "w") as f:
        json.dump(preds, f, indent=2)
    print(f"Saved to {out_dir / 'zero_shot_predictions.json'}")

    # ---- figure: uniform square tiles, caption below each, drawn at printed size ----
    fig_w, fs = 6.5, 7                                   # width (in), font size (pt)
    n = len(image_paths)
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    col_w = fig_w / ncols
    wrap = max(16, int(col_w * 72 / (fs * 0.75)))        # chars per line that fit a column
    fig, axes = plt.subplots(nrows * 2, ncols, figsize=(fig_w, 2.5 * nrows),
                             gridspec_kw={"height_ratios": [1, 0.35] * nrows})
    axes = axes.reshape(nrows * 2, ncols)
    for ax in axes.flat:
        ax.axis("off")
    caps = list(preds.values()) if len(preds) == n else [preds.get(p, "") for p in image_paths]
    for i, p in enumerate(image_paths):
        r, c = (i // ncols) * 2, i % ncols
        img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        axes[r, c].imshow(ImageOps.fit(img, (400, 400)))
        axes[r + 1, c].text(0.5, 1, textwrap.fill(preds.get(p, caps[i]), wrap),
                            va="top", ha="center", fontsize=fs)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.06, hspace=0.03)
    fig.savefig(out_dir / "zero_shot.png", dpi=250, bbox_inches="tight", pad_inches=0.03)
    plt.show()
    print(f"Saved figure to {out_dir / 'zero_shot.png'}")
    return preds

In [ ]:
# --------------------------------------------------------
# 7. EXTRA ANALYSIS: attention, prompts, model comparison
# --------------------------------------------------------
def _fname(key):
    """File name from a saved prediction key (handles both \\ and / separators)."""
    return key.replace("\\", "/").split("/")[-1]
 
 
def _load_test_refs():
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs = {}
    for a in coco["annotations"]:
        refs.setdefault(a["image_id"], []).append(a["caption"].strip())
    return root, id_to_file, refs
 
 
def _sample_test_images(n_samples):
    """Fixed-seed sample of test images (same seed -> same images every call)."""
    root, id_to_file, refs = _load_test_refs()
    ids = random.Random(CONFIG["seed"]).sample(list(refs.keys()), n_samples)
    paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in ids]
    return ids, paths, refs
 
 
def visualize_attention(checkpoint_path=None, n_samples=3, max_words=8, layer=-1, num_beams=3):
    """
    Word-level attention for GIT. GIT has no cross-attention: image patches and
    caption words sit in ONE decoder sequence. For each generated word we take
    the decoder's attention from that word's position to the 14x14 image patches
    (averaged over heads, from decoder layer `layer`) and overlay it on the image.
    Saves outputs/<checkpoint>/attention_words.png
    """
    import numpy as np
    import matplotlib.pyplot as plt
 
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    # "eager" attention is needed so attention probabilities are actually computed
    try:
        model, processor = load_model_and_processor(model_path, attn_implementation="eager")
    except (TypeError, ValueError):
        model, processor = load_model_and_processor(model_path)
    model.eval()
    tok = processor.tokenizer
    mean = torch.tensor(processor.image_processor.image_mean).view(3, 1, 1)
    std = torch.tensor(processor.image_processor.image_std).view(3, 1, 1)
 
    _, paths, _ = _sample_test_images(n_samples)
 
    # Capture the attention probabilities of one decoder layer with a hook.
    store = {}
    def hook(module, inputs, output):
        # Attention probabilities are the only 4-D tensor: (batch, heads, query, key).
        if isinstance(output, tuple):
            for item in output:
                if torch.is_tensor(item) and item.ndim == 4:
                    store["attn"] = item.detach().float().cpu()
                    break
    handle = model.git.encoder.layer[layer].attention.self.register_forward_hook(hook)
 
    fig, axes = plt.subplots(n_samples, 1 + max_words,
                             figsize=(2.6 * (1 + max_words), 2.9 * n_samples))
    axes = np.atleast_2d(axes)
    try:
        for r, path in enumerate(paths):
            image = Image.open(path).convert("RGB")
            pv = processor(images=image, return_tensors="pt").pixel_values.to(device)
            with torch.no_grad():
                ids = model.generate(pixel_values=pv, max_length=CONFIG["max_caption_len"],
                                     num_beams=num_beams)
                store.clear()
                try:
                    model(pixel_values=pv, input_ids=ids, output_attentions=True, use_cache=False)
                except TypeError:
                    model(pixel_values=pv, input_ids=ids)
            if "attn" not in store:
                raise RuntimeError(
                    "Could not capture attention weights (your transformers version may not "
                    "return them for GIT). Send me `pip show transformers` and I'll adapt this.")
 
            attn = store["attn"][0].mean(0)                 # (seq, seq), averaged over heads
            n_img = attn.shape[-1] - ids.shape[1]           # image tokens: 197 = CLS + 14x14
            tokens = tok.convert_ids_to_tokens(ids[0].tolist())
            caption = tok.decode(ids[0], skip_special_tokens=True)
 
            # Show the image exactly as the model saw it (224x224 centre crop).
            base = (pv[0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
            axes[r, 0].imshow(base)
            axes[r, 0].axis("off")
            axes[r, 0].set_title(textwrap.fill(caption, 28), fontsize=7)
 
            shown = 0
            for k in range(1, len(tokens)):
                if tokens[k] in ("[SEP]", "[PAD]") or shown >= max_words:
                    break
                # the row at position k-1 is what predicts token k
                patches = attn[n_img + k - 1, :n_img][1:]   # drop the image CLS token
                side = int(round(patches.numel() ** 0.5))
                heat = patches[: side * side].reshape(1, 1, side, side)
                heat = F.interpolate(heat, size=base.shape[:2], mode="bilinear",
                                     align_corners=False)[0, 0].numpy()
                heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
                ax = axes[r, 1 + shown]
                ax.imshow(base)
                ax.imshow(heat, cmap="jet", alpha=0.5)
                ax.axis("off")
                ax.set_title(tokens[k], fontsize=9)
                shown += 1
            for c in range(1 + shown, 1 + max_words):
                axes[r, c].axis("off")
    finally:
        handle.remove()
 
    out_dir = get_output_dir(checkpoint_path)
    fig.tight_layout()
    fig.savefig(out_dir / "attention_words.png", dpi=150)
    print(f"Saved word-level attention to {out_dir / 'attention_words.png'}")
 


def prompted_captioning(checkpoint_path=None, prompts=None, n_samples=3):
    """
    Prompt-conditioned captioning: force the caption to start with a text prefix.
    Compares unprompted vs prompted captions on the same fixed test images.
    Figure is drawn at its printed size (6.5 in wide) so fonts stay readable in the report.
    """
    import matplotlib.pyplot as plt

    prompts = prompts or ["a photography of", "a close up of", "two people"]
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    model, processor = load_model_and_processor(model_path)
    model.eval()
    tok = processor.tokenizer

    _, paths, _ = _sample_test_images(n_samples)
    images = [Image.open(p).convert("RGB") for p in paths]
    pv = processor(images=images, return_tensors="pt").pixel_values.to(device)

    def run(prompt=None):
        kwargs = {}
        if prompt:
            ids = [tok.cls_token_id] + tok(prompt, add_special_tokens=False).input_ids
            kwargs["input_ids"] = torch.tensor([ids] * len(images), device=device)
        with torch.no_grad():
            out = model.generate(pixel_values=pv, max_length=CONFIG["max_caption_len"],
                                 num_beams=4, **kwargs)
        return [c.strip() for c in tok.batch_decode(out, skip_special_tokens=True)]

    results = {"unprompted": run(None)}
    for p in prompts:
        results[p] = run(p)

    # ---- figure: drawn at printed size so text stays readable ----
    fig_w, fs = 6.5, 7                                   # width (in), font size (pt)
    col_w = fig_w / n_samples
    wrap = max(16, int(col_w * 72 / (fs * 0.75)))        # chars per line that fit a column
    fig, axes = plt.subplots(2, n_samples, figsize=(fig_w, 3.8),
                             gridspec_kw={"height_ratios": [1, 1]})
    if n_samples == 1:
        axes = axes.reshape(2, 1)
    for col, img in enumerate(images):
        img = ImageOps.exif_transpose(img).convert("RGB")
        axes[0, col].imshow(ImageOps.fit(img, (400, 400)))
        axes[0, col].axis("off")
        lines = []
        for name, caps in results.items():
            label = "no prompt" if name == "unprompted" else f'"{name}"'
            lines.append(textwrap.fill(f"[{label}] {caps[col]}", wrap))
        axes[1, col].text(0, 1, "\n".join(lines), va="top", ha="left", fontsize=fs)
        axes[1, col].axis("off")

    out_dir = get_output_dir(checkpoint_path)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.06, hspace=0.03)
    fig.savefig(out_dir / "prompted_captions.png", dpi=250, bbox_inches="tight", pad_inches=0.03)
    with open(out_dir / "prompted_captions.json", "w") as f:
        json.dump({"images": paths, "captions": results}, f, indent=2)
    print(f"Saved prompted captions to {out_dir}")
    return results


def compare_predictions(pred_files, n_samples=3):
    """
    Side-by-side captions from several models on the SAME test images, built from
    the predictions.json files that evaluate() already saved (no model loading).
    Figure is drawn at its printed size (6.5 in wide) so fonts stay readable in the report.
    """
    import matplotlib.pyplot as plt

    preds = {}
    for name, p in pred_files.items():
        with open(p) as f:
            preds[name] = {_fname(k): v for k, v in json.load(f).items()}
    common = sorted(set.intersection(*[set(d) for d in preds.values()]))
    chosen = random.Random(CONFIG["seed"]).sample(common, n_samples)

    root, id_to_file, refs = _load_test_refs()
    name_to_id = {v: k for k, v in id_to_file.items()}

    fig_w, fs = 6.5, 7                                   # width (in), font size (pt)
    ncols = min(n_samples, 3)
    nrows = (n_samples + ncols - 1) // ncols
    col_w = fig_w / ncols
    wrap = max(16, int(col_w * 72 / (fs * 0.75)))        # chars per line that fit a column
    fig, axes = plt.subplots(nrows * 2, ncols, figsize=(fig_w, 3.8 * nrows),
                             gridspec_kw={"height_ratios": [1, 1] * nrows})
    axes = axes.reshape(nrows * 2, ncols)
    for ax in axes.flat:
        ax.axis("off")
    for i, fname in enumerate(chosen):
        r, c = (i // ncols) * 2, i % ncols
        img = ImageOps.exif_transpose(Image.open(root / CONFIG["test_img_dir"] / fname)).convert("RGB")
        axes[r, c].imshow(ImageOps.fit(img, (400, 400)))
        lines = [textwrap.fill(f"GT: {refs[name_to_id[fname]][0]}", wrap)]
        for name, d in preds.items():
            lines.append(textwrap.fill(f"{name}: {d[fname]}", wrap))
        axes[r + 1, c].text(0, 1, "\n".join(lines), va="top", ha="left", fontsize=fs)

    out_dir = Path(CONFIG["output_dir"]) / "comparison"
    out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
prepare_data()

In [ ]:
evaluate(checkpoint_path="microsoft/git-base")  

In [ ]:
evaluate(checkpoint_path="microsoft/git-base-coco")

In [ ]:
qualitative_analysis(checkpoint_path="microsoft/git-base-coco")

In [ ]:
tune_hyperparameters(n_trials=12)

In [ ]:
train(run_name="git_1epoch", num_epochs=1)

In [ ]:
train(run_name="gitbase_1epoch", num_epochs=1)

In [ ]:
evaluate("./checkpoints/gitbase_1epoch/epoch0")

In [ ]:
visualize_attention(checkpoint_path="./checkpoints/gitbase_1epoch/epoch0", n_samples=3, max_words=8, layer=-1)

In [ ]:
prompted_captioning(checkpoint_path="./checkpoints/gitbase_1epoch/epoch0", prompts=["a photography of", "a close up of", "two people"], n_samples=4)


In [ ]:
zero_shot(image_dir="./image_dir", checkpoint_path="./checkpoints/gitbase_1epoch/epoch0")

In [ ]:
train(run_name="gitbase_3epoch", num_epochs=3)

In [ ]:
evaluate("./checkpoints/gitbase_3epoch/epoch2")

In [ ]:
compare_predictions({
    "GIT-base (untuned)":   "outputs/git-base/predictions.json",
    "GIT-base fine-tuned":  "outputs/gitbase_1epoch_epoch0/predictions.json",
    "GIT-base-coco":        "outputs/git-base-coco/predictions.json",
    "BLIP (untuned)":       "outputs/blip-base-coco/predictions.json",
    "BLIP fine-tuned":      "outputs/blip_1epoch_epoch0/predictions.json",
}, n_samples=3)